# 项目一 · 任务一　用 KNN 推荐车型

## 基础项目　用三项尺寸推荐 GoF

### 1. 明确预测任务

某汽车网站记录了一位用户对车型的浏览行为。用户点开查看的车型记为「浏览」，直接略过的车型记为「略过」。现在网站想判断是否向这位用户推荐 GoF。我们用车长、车宽和车高比较车型，从四辆训练车中找出最接近 GoF 的三辆，再根据用户对这些车的历史行为投票，给出「推荐」或「不推荐」。

#### 项目目标

给出 GoF 的推荐结论，再用两辆有历史记录的测试车检验预测。完成后，你应能说出谁参加了投票，以及测试准确率是怎么计算的。

#### 已知数据

| 车型 | 车长（mm） | 车宽（mm） | 车高（mm） | 历史行为 | 数据用途 |
| --- | ---: | ---: | ---: | --- | --- |
| BAOJ310 | 4032 | 1680 | 1450 | 略过 | 训练 |
| XW-M1 | 4330 | 1535 | 1885 | 浏览 | 训练 |
| POL | 4053 | 1740 | 1449 | 略过 | 训练 |
| BW5 | 5087 | 1868 | 1500 | 浏览 | 训练 |
| BDe6 | 4560 | 1822 | 1645 | 浏览 | 测试 |
| Wuling | 3797 | 1510 | 1820 | 略过 | 测试 |
| GoF | 4259 | 1799 | 1452 | 待预测 | 实际预测 |

训练车提供用于投票的已知标签。BDe6 和 Wuling 的历史行为用于核对预测结果。GoF 是网站当前要推荐的车型，表中只有它的特征。

#### 分析步骤

1. 按车型整理特征和历史行为，把历史行为编码成分类标签。
2. 取 $K=3$，用 Python 计算欧氏距离，排序、选取近邻并投票，给出 GoF 的预测结果。
3. 对两辆测试车做同样的预测，逐辆核对历史标签，计算准确率。
4. 用 `KNeighborsClassifier` 复现 GoF 的预测，并在相同的测试车上检验结果。

在 Python 3 内核中从上到下运行代码。第 5 节和扩展项目使用 scikit-learn。

### 2. 准备特征与标签

`X_train` 和 `X_test` 分别存放训练车、测试车的特征。外层列表表示有多辆车，内层列表表示一辆车的车长、车宽、车高。比如，`X_train[0]` 是第一辆车的尺寸，`X_train[0][1]` 是这辆车的车宽。`x_gof` 只存放一辆车的三个尺寸。

先把特征和历史行为按数据表的顺序录入，下一步再编码标签。

In [25]:
train_names = ["BAOJ310", "XW-M1", "POL", "BW5"]
test_names = ["BDe6", "Wuling"]

X_train = [
    [4032, 1680, 1450],  # BAOJ310
    [4330, 1535, 1885],  # XW-M1
    [4053, 1740, 1449],  # POL
    [5087, 1868, 1500],  # BW5
]
X_test = [
    [4560, 1822, 1645],  # BDe6
    [3797, 1510, 1820],  # Wuling
]
x_gof = [4259, 1799, 1452]

train_behaviors = ["略过", "浏览", "略过", "浏览"]
test_behaviors = ["浏览", "略过"]
k = 3

In [26]:
x_gof

[4259, 1799, 1452]

#### 把历史行为编码成标签

用字典规定「略过」对应 `0`、「浏览」对应 `1`。训练标签存到 `y_train`，测试标签存到 `y_test`。每个标签都与同一位置的车型特征对应。

用数字编码后，可以直接统计各类别的票数，也方便把预测标签与真实标签逐一比较。这两个数字是类别代号。计算车型距离时只用特征列。

In [24]:
behavior_to_label = {"略过": 0, "浏览": 1}
label_to_recommendation = {0: "不推荐", 1: "推荐"}

y_train = []
for behavior in train_behaviors:
    y_train.append(behavior_to_label[behavior])

y_test = []
for behavior in test_behaviors:
    y_test.append(behavior_to_label[behavior])

print("训练标签", y_train)
print("测试标签", y_test)

训练标签 [0, 1, 0, 1]
测试标签 [1, 0]


### 3. 用 Python 逐步实现 KNN

**计算距离 → 按距离排序 → 选出最近的 $K$ 辆车 → 根据近邻标签投票**

#### 3.1 计算距离

先只计算 GoF 到 POL 的距离。用两辆车对应尺寸的差值，代入欧氏距离公式。

$$
d = \sqrt{(4259-4053)^2 + (1799-1740)^2 + (1452-1449)^2}
  \approx 214.3\ \mathrm{mm}.
$$

POL 在训练列表的第三行，Python 从 `0` 开始编号，所以用 `X_train[2]` 取出它。对一辆车的尺寸列表，索引 `0`、`1`、`2` 分别对应车长、车宽和车高。

In [3]:
from math import sqrt

pol = X_train[2]
length_diff = x_gof[0] - pol[0]
width_diff = x_gof[1] - pol[1]
height_diff = x_gof[2] - pol[2]
distance_to_pol = sqrt(length_diff**2 + width_diff**2 + height_diff**2)

print(f"GoF 到 POL 的距离为 {distance_to_pol:.1f} mm")

GoF 到 POL 的距离为 214.3 mm


接下来，用 `for` 循环依次取出四辆训练车，对每辆车重复同一个计算。`range(len(X_train))` 给出四辆训练车的索引。每算完一辆，就用 `append()` 把「距离、车型、标签」这条记录加到结果列表末尾。

In [4]:
distance_rows = []

for i in range(len(X_train)):
    car = X_train[i]
    length_diff = x_gof[0] - car[0]
    width_diff = x_gof[1] - car[1]
    height_diff = x_gof[2] - car[2]
    distance_mm = sqrt(length_diff**2 + width_diff**2 + height_diff**2)

    distance_rows.append([distance_mm, train_names[i], y_train[i]])
    print(f"{train_names[i]}，{distance_mm:.1f} mm，标签 {y_train[i]}")

BAOJ310，256.3 mm，标签 0
XW-M1，512.1 mm，标签 1
POL，214.3 mm，标签 0
BW5，832.3 mm，标签 1


#### 把距离公式封装成函数

距离计算用到两辆车的尺寸，计算结果是一个距离。把公式写成 `distance(car_a, car_b)` 后，循环每次只需传入两辆车。`return` 把算好的距离交回循环，供下一行保存。

下面用函数再算一次四辆车的距离，并与刚才的列表核对。

In [5]:
def distance(car_a, car_b):
    # 依次计算车长、车宽和车高的差值。
    length_diff = car_a[0] - car_b[0]
    width_diff = car_a[1] - car_b[1]
    height_diff = car_a[2] - car_b[2]
    return sqrt(length_diff**2 + width_diff**2 + height_diff**2)

distance_rows_from_function = []
for i in range(len(X_train)):
    distance_mm = distance(x_gof, X_train[i])
    distance_rows_from_function.append([distance_mm, train_names[i], y_train[i]])

print("distance() 函数计算的距离记录")
for i in range(len(distance_rows_from_function)):
    print(distance_rows_from_function[i])


distance() 函数计算的距离记录
[256.3084079775769, 'BAOJ310', 0]
[512.0800718637662, 'XW-M1', 1]
[214.30352306950067, 'POL', 0]
[832.2553694630033, 'BW5', 1]


#### 3.2 按距离排序

`sorted()` 返回一个按升序排列的新列表，原列表仍保留原来的顺序。比较两条列表记录时，Python 先比较第一项。这里把距离放在第一项，因此较近的车排在前面。每条记录中的车型和标签随距离一起移动。

运行后，对照排序前后的输出。哪辆车从第三行移到了第一行？

In [6]:
sorted([5,3,2,1,4])

[1, 2, 3, 4, 5]

In [7]:
sorted([[5, 7], [2, 1], [4, 6]])

[[2, 1], [4, 6], [5, 7]]

In [8]:
sorted([[5, 7], [2, 1], [4, 6]], key=lambda x: x[0], reverse=True)

[[5, 7], [4, 6], [2, 1]]

In [9]:
sorted_rows = sorted(distance_rows_from_function)

print("排序前（车型、距离、标签）")
for row in distance_rows_from_function:
    print(f"{row[1]}，{row[0]:.1f} mm，标签 {row[2]}")

print()
print("排序后（车型、距离、标签）")
for row in sorted_rows:
    print(f"{row[1]}，{row[0]:.1f} mm，标签 {row[2]}")

排序前（车型、距离、标签）
BAOJ310，256.3 mm，标签 0
XW-M1，512.1 mm，标签 1
POL，214.3 mm，标签 0
BW5，832.3 mm，标签 1

排序后（车型、距离、标签）
POL，214.3 mm，标签 0
BAOJ310，256.3 mm，标签 0
XW-M1，512.1 mm，标签 1
BW5，832.3 mm，标签 1


#### 3.3 选出最近的三个样本

`[:k]` 表示取列表的前 `k` 条。先看排序后的结果，指出哪三辆车将参加投票，再运行代码核对。

In [10]:
neighbors = sorted_rows[:k]

for row in neighbors:
    print(f"近邻 {row[1]}，标签 {row[2]}")

近邻 POL，标签 0
近邻 BAOJ310，标签 0
近邻 XW-M1，标签 1


#### 3.4 投票并给出预测

1. 遍历三个近邻，用 `row[2]` 取出每条记录的标签。GoF 的三个近邻给出 `[0, 0, 1]`。
2. `count(0)` 和 `count(1)` 分别数出略过和浏览的票数。每辆近邻只有一票，因此两类票数相加等于 `3`。
3. 用 `if` 和 `else` 判断哪一类票数较多，再把结果换成「推荐」或「不推荐」。本例有两个类别、三张等权选票，因此总有一类票数较多。

In [11]:
[0, 1, 0, 1, 1].count(1)

3

In [12]:
neighbor_labels = []
for row in neighbors:
    neighbor_labels.append(row[2])

votes_0 = neighbor_labels.count(0)
votes_1 = neighbor_labels.count(1)

if votes_1 > votes_0:
    gof_manual = 1
else:
    gof_manual = 0

print("近邻标签", neighbor_labels)
print(f"略过 {votes_0} 票，浏览 {votes_1} 票")
print(f"GoF：{label_to_recommendation[gof_manual]}（{gof_manual}）")

近邻标签 [0, 0, 1]
略过 2 票，浏览 1 票
GoF：不推荐（0）


### 4. 用测试车检验逐步实现的结果

GoF 的预测展示了 KNN 怎样给出推荐。为了检查预测是否正确，我们使用两辆有真实标签的测试车。每次把一辆测试车当作新的输入，仍然只从四辆训练车中找近邻。

下面的 `predict_one()` 把刚才的四步放在一起。输入一辆车的尺寸、训练数据和近邻数，返回一个预测标签。随后对每辆测试车调用一次这个函数。

准确率的分子是预测标签与真实标签相同的车辆数，分母是测试车辆总数。

$$
\text{准确率} = \frac{\text{预测正确的测试车辆数}}{\text{测试车辆总数}}.
$$

In [13]:
def predict_one(car, X_train, y_train, k):
    # 计算距离，记录距离与训练车索引。
    rows = []
    for i in range(len(X_train)):
        rows.append([distance(car, X_train[i]), i])

    # 排序并取出前 k 条记录，再按索引取对应标签。
    nearest_rows = sorted(rows)[:k]
    labels = []
    for row in nearest_rows:
        labels.append(y_train[row[1]])

    # 本例取 k=3，按两个类别的票数得出预测。
    if labels.count(1) > labels.count(0):
        return 1
    return 0

manual_test_predictions = []
for car in X_test:
    manual_test_predictions.append(predict_one(car, X_train, y_train, k))

把逐辆核对和计算准确率写成 `report_accuracy()`。这样后面的两次模型评估也能使用同一段计算代码。

In [14]:
def report_accuracy(names, actual_labels, predicted_labels):
    correct_count = 0
    for i in range(len(actual_labels)):
        actual = actual_labels[i]
        predicted = predicted_labels[i]
        if predicted == actual:
            correct_count += 1
        print(f"{names[i]}，预测 {predicted}，实际 {actual}")

    total = len(actual_labels)
    accuracy = correct_count / total
    print(f"预测正确 {correct_count}/{total}，准确率 {accuracy:.0%}")
    return accuracy

manual_accuracy = report_accuracy(test_names, y_test, manual_test_predictions)

BDe6，预测 1，实际 1
Wuling，预测 0，实际 0
预测正确 2/2，准确率 100%


### 5. 用 scikit-learn 完成同一个任务

#### 创建分类器

| 本例的设置 | 它决定什么 |
|---|---|
| `n_neighbors=3` | 取最近的三辆训练车投票。 |
| `metric="euclidean"` | 用前面算过的欧氏距离比较车型。 |
| `weights="uniform"` | 每辆近邻的票权相同，各投一票。 |

#### 提供训练数据并预测

- `fit(X_train, y_train)` 接收四辆历史车的特征和标签，分类器用这批带标签的样本准备后续的近邻查询。
- `predict([x_gof])` 接收待预测车型的特征，从训练车中找近邻，再按已知标签投票。外层方括号表示本次预测包含一行数据，`[0]` 从返回结果取出这辆车的预测标签。
- `predict(X_test)` 一次接收两辆测试车的特征，按输入顺序返回两个标签。得到结果后，再用 `y_test` 核对。

In [15]:
from sklearn.neighbors import KNeighborsClassifier

model = KNeighborsClassifier(n_neighbors=k, metric="euclidean", weights="uniform")
model.fit(X_train, y_train)

gof_sklearn = model.predict([x_gof])[0]
print(f"GoF：{label_to_recommendation[gof_sklearn]}（{gof_sklearn}）")
print("两种实现的 GoF 预测一致", gof_sklearn == gof_manual)

GoF：不推荐（0）
两种实现的 GoF 预测一致 True


In [16]:
sklearn_test_predictions = model.predict(X_test)
sklearn_accuracy = report_accuracy(test_names, y_test, sklearn_test_predictions)

BDe6，预测 1，实际 1
Wuling，预测 0，实际 0
预测正确 2/2，准确率 100%


## 扩展项目　加入油耗和售价后推荐车型

### 1. 明确目标与数据

如果还知道每辆车的油耗和售价，就可以用五项特征比较车型。车长以毫米计，售价以万元计。若两辆车的车长相差 $100$ mm、售价相差 $5$ 万元，直接代入距离公式，两项平方分别是 $100^2$ 和 $5^2$。因此，计算距离前先按列做 Min–Max 归一化。

#### 项目目标

用归一化后的五项特征训练 KNN，检验模型在 BDe6 和 Wuling 上的结果，再用这个模型判断是否推荐历史行为未知的 CRV。

#### 已知数据

| 车型 | 车长（mm） | 车宽（mm） | 车高（mm） | 油耗（L/100 km） | 售价（万元） | 历史行为 | 数据用途 |
| --- | ---: | ---: | ---: | ---: | ---: | --- | --- |
| BAOJ310 | 4032 | 1680 | 1450 | 5.3 | 5.6 | 略过 | 训练 |
| XW-M1 | 4330 | 1535 | 1885 | 7.8 | 14.5 | 浏览 | 训练 |
| POL | 4053 | 1740 | 1449 | 6.2 | 10.8 | 略过 | 训练 |
| BW5 | 5087 | 1868 | 1500 | 8.5 | 25.6 | 浏览 | 训练 |
| BDe6 | 4560 | 1822 | 1645 | 7.8 | 15.8 | 浏览 | 测试 |
| Wuling | 3797 | 1510 | 1820 | 5.5 | 9.6 | 略过 | 测试 |

#### 分析步骤

1. 用四辆训练车确定五列特征各自的最小值和最大值。
2. 用同一组参数转换训练车和测试车，在缩放后的数据上拟合分类器、预测并计算准确率。
3. 独立完成 CRV 的特征转换和分类预测。

In [17]:
# 五列的顺序是车长、车宽、车高、油耗、售价。
X_train_ext = [
    [4032, 1680, 1450, 5.3, 5.6],  # BAOJ310
    [4330, 1535, 1885, 7.8, 14.5],  # XW-M1
    [4053, 1740, 1449, 6.2, 10.8],  # POL
    [5087, 1868, 1500, 8.5, 25.6],  # BW5
]

X_test_ext = [
    [4560, 1822, 1645, 7.8, 15.8],  # BDe6
    [3797, 1510, 1820, 5.5, 9.6],  # Wuling
]

### 2. 用训练数据确定归一化参数

每一列单独计算最小值和最大值。训练车、测试车和稍后要预测的 CRV 都使用同一组参数转换。

$$
x_{\text{缩放}}
= \frac{x-x_{\min,\text{训练}}}{x_{\max,\text{训练}}-x_{\min,\text{训练}}}.
$$

以车宽为例，四辆训练车的范围是 1535–1868 mm。Wuling 的车宽是 1510 mm，转换后是 $(1510-1535)/(1868-1535)\approx -0.075$。当测试值低于训练集最小值时，缩放结果可以小于 $0$。

`scaler.fit()` 确定每一列的缩放参数。`scaler.transform()` 按这些参数转换输入的数据。

In [18]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
scaler.fit(X_train_ext)

X_train_scaled = scaler.transform(X_train_ext)
X_test_scaled = scaler.transform(X_test_ext)

print(f"训练车的车宽范围为 {scaler.data_min_[1]:.0f}–{scaler.data_max_[1]:.0f} mm")
print(f"Wuling 的车宽归一化结果为 {X_test_scaled[1][1]:.3f}")

训练车的车宽范围为 1535–1868 mm
Wuling 的车宽归一化结果为 -0.075


### 3. 预测测试车并核对结果

创建一个新分类器，仍然取三个近邻、使用欧氏距离和等权投票。用缩放后的训练特征 `X_train_scaled` 和原来的 `y_train` 拟合，再预测缩放后的两辆测试车。最后沿用基础项目的 `report_accuracy()` 计算准确率。

In [19]:
model_ext = KNeighborsClassifier(n_neighbors=k, metric="euclidean", weights="uniform")
model_ext.fit(X_train_scaled, y_train)

ext_test_predictions = model_ext.predict(X_test_scaled)
ext_accuracy = report_accuracy(test_names, y_test, ext_test_predictions)

BDe6，预测 1，实际 1
Wuling，预测 0，实际 0
预测正确 2/2，准确率 100%


两套特征方案在这两辆测试车上的准确率相同。观察 Wuling 的最近邻，使用三项尺寸时它最接近 BAOJ310，使用归一化后的五项特征时它最接近 XW-M1。近邻排序可以改变，多数票结果仍可能相同。判断哪套方案在实际推荐中更可靠，还需要更多有标签的测试数据。

### 4. 独立上机　预测是否推荐 CRV

现在网站需要判断是否向这位用户推荐 CRV。根据下面的数据，使用刚刚准备好的归一化工具和分类器完成预测。

| 车型 | 车长（mm） | 车宽（mm） | 车高（mm） | 油耗（L/100 km） | 售价（万元） | 历史行为 |
|---|---:|---:|---:|---:|---:|---|
| CRV | 4585 | 1855 | 1679 | 7.6 | 19.6 | 待预测 |

1. 把五项特征按表中顺序写成一行五列，参考基础项目中预测 GoF 时的两层方括号。
2. 调用 `scaler.transform()`，用训练车确定的参数转换 CRV 的特征。
3. 把转换结果交给 `model_ext.predict()`。取出预测标签，说明是否推荐 CRV。

In [20]:
# ① 在这里写出 CRV 的一行五列原始特征。
# ② 用已经拟合的 scaler 转换这行数据。
# ③ 调用 model_ext.predict()，打印标签和推荐结论。

数据来源为项目内的《原始-基础数据.csv》和《原始-扩展数据.csv》。本 Notebook 直接列出了所用车型的数据。